In [22]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Đọc dữ liệu
df = pd.read_csv('Agri_Data_Cleaned.csv')

# 2. Lấy các cột dạng số (loại trừ cột mục tiêu 'Yield' không cần chuẩn hóa)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
cols_to_scale = [col for col in numeric_cols if col != 'Yield']

# 3. Khởi tạo thuật toán MinMaxScaler
scaler = MinMaxScaler()

# 4. Tạo DataFrame mới để lưu kết quả và tiến hành biến đổi (Scale)
df_scaled = df.copy()
df_scaled[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# 5. Lưu ra file CSV
# Sử dụng float_format='%.15f' để đảm bảo ghi đủ tối đa 15 chữ số thập phân
output_filename = 'Agri_Data_Scaled_HighPrecision.csv'
df_scaled.to_csv(output_filename, index=False, float_format='%.15f')

print("Đã lưu thành công file với độ chính xác cao!")

Đã lưu thành công file với độ chính xác cao!


In [23]:
import pandas as pd
import numpy as np
import networkx as nx

# 1. Đọc dữ liệu (Đã chuẩn hóa 0-1)
df = pd.read_csv('Agri_Data_Scaled_HighPrecision.csv')

# Tính ma trận tương quan cho các biến số
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Yield' in numeric_cols: numeric_cols.remove('Yield')

corr_matrix = df[numeric_cols].corr()

# 2. Xây dựng đồ thị tìm các nhóm biến tương quan cực cao (>0.85)
threshold = 0.8
G = nx.Graph()

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        col1 = corr_matrix.columns[i]
        col2 = corr_matrix.columns[j]
        if abs(corr_matrix.iloc[i, j]) > threshold:
            G.add_edge(col1, col2)

connected_components = list(nx.connected_components(G))

# Hàm ghép tên tự động thông minh bằng cách nối từ khóa chung
def generate_meaningful_name(columns):
    words_list = [col.replace('_', ' ').split() for col in columns]
    common_words = []
    if len(words_list) > 1:
        for bw in words_list[0]:
            if all(bw in wl for wl in words_list[1:]): common_words.append(bw)
            
    if common_words:
        prefix = "_".join(common_words)
        unique_parts = ["".join([w for w in wl if w not in common_words]) for wl in words_list]
        return f"Combined_{prefix}_" + "_".join([p for p in unique_parts if p])
    else:
        return "Combined_" + "_".join([col.replace(' ', '')[:10] for col in columns])

# 3. Tiến hành gộp và loại bỏ để giảm tương quan
df_engineered = df.copy()
cols_to_drop = []

for idx, comp in enumerate(connected_components):
    cols_in_comp = list(comp)
    new_col_name = generate_meaningful_name(cols_in_comp)
    
    # Lấy trung bình (để gom đặc trưng của 2-3 cột cùng lúc)
    df_engineered[new_col_name] = df_engineered[cols_in_comp].mean(axis=1)
    cols_to_drop.extend(cols_in_comp)

# 4. Xóa bỏ các cột cũ gây nhiễu/tương quan
df_engineered.drop(columns=list(set(cols_to_drop)), inplace=True)

# Xuất dữ liệu
df_engineered.to_csv('Agri_Data_Engineered_Features.csv', index=False, float_format='%.15f')

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- CẤU HÌNH: CÁC NHÓM CỘT CẦN GỘP ---
# Dựa trên phân tích VIF và kiến thức nông nghiệp
GROUPS_TO_COMBINE = {
    # Nhóm 1: Kết cấu đất (Sand + Silt + Clay = 100%) -> Gom thành chỉ số kết cấu
    'Soil_Texture_Index': ['Sand', 'Silt', 'Clay'],
    
    # Nhóm 2: Hóa lý đất (Dinh dưỡng + Độ xốp)
    'Soil_PhysioChem_Index': ['Nitrogen', 'Organic_Carbon', 'CN_Ratio', 'Bulk_Density'],
    
    # Nhóm 3: Sức khỏe cây trồng (Các chỉ số NDVI, EVI tương quan rất mạnh)
    'Vegetation_Health_Index': [
        'EVI', 
        'NDVI_Season_Min', 
        'NDVI_Season_CV', 
        'Combined_NDVI_Seaso_LAI_NDVI_Seaso_FPAR', 
        'Combined_NDVI_Season_Range_Std'
    ],
    
    # Nhóm 4: Gió (Trung bình & Cực đại)
    'Wind_Index': ['Wind_Mean', 'Wind_Max'],
    
    # Nhóm 5: Môi trường đất (Độ ẩm & Nhiệt độ bề mặt thường đi đôi với nhau)
    'Soil_Environment_Index': ['Soil_Moisture_mm', 'LST_C']
}

def calculate_vif(data):
    """Hàm tính VIF cho dataframe"""
    # Chỉ lấy cột số
    data_numeric = data.select_dtypes(include=[np.number])
    # Xử lý vô cực và NaN
    data_numeric = data_numeric.replace([np.inf, -np.inf], np.nan).fillna(data_numeric.mean())
    
    vif_data = pd.DataFrame()
    vif_data["Feature"] = data_numeric.columns
    vif_data["VIF"] = [variance_inflation_factor(data_numeric.values, i) 
                       for i in range(len(data_numeric.columns))]
    return vif_data.sort_values(by="VIF", ascending=False)

def apply_pca_transform(df, group_name, cols):
    """Hàm áp dụng PCA để gộp cột"""
    # Kiểm tra xem các cột có tồn tại trong dữ liệu không
    existing_cols = [c for c in cols if c in df.columns]
    
    if len(existing_cols) < 2:
        print(f"Bỏ qua nhóm {group_name}: Không đủ cột thành phần.")
        return df
    
    print(f"Đang xử lý nhóm: {group_name} <- {existing_cols}")
    
    # Lấy dữ liệu và chuẩn hóa (Standardize)
    sub_data = df[existing_cols].fillna(df[existing_cols].mean())
    scaler = StandardScaler()
    sub_data_scaled = scaler.fit_transform(sub_data)
    
    # Áp dụng PCA để lấy thành phần chính (giữ lại thông tin quan trọng nhất)
    pca = PCA(n_components=1)
    new_col_values = pca.fit_transform(sub_data_scaled)
    
    # Tạo cột mới và xóa cột cũ
    df[group_name] = new_col_values
    df.drop(columns=existing_cols, inplace=True)
    
    return df

# --- CHẠY CHƯƠNG TRÌNH ---
# 1. Load dữ liệu
file_path = 'Agri_Data_Engineered_Features.csv'  # Đổi tên file nếu cần
df = pd.read_csv(file_path)

print("--- Bắt đầu xử lý giảm đa cộng tuyến (VIF) ---")

# 2. Áp dụng gộp cột
df_clean = df.copy()
for new_col_name, columns_to_merge in GROUPS_TO_COMBINE.items():
    df_clean = apply_pca_transform(df_clean, new_col_name, columns_to_merge)

# 3. Kiểm tra lại VIF
print("\n--- Đang tính toán VIF sau khi xử lý ---")
final_vif = calculate_vif(df_clean)
print(final_vif.head(10))  # In top 10 VIF cao nhất

# 4. Lưu file
output_file = 'Agri_Data_VIF_Fixed_Clean.csv'
df_clean.to_csv(output_file, index=False)
print(f"\nĐã lưu file kết quả tại: {output_file}")

--- Bắt đầu xử lý giảm đa cộng tuyến (VIF) ---
Đang xử lý nhóm: Soil_Texture_Index <- ['Sand', 'Silt', 'Clay']
Đang xử lý nhóm: Soil_PhysioChem_Index <- ['Nitrogen', 'Organic_Carbon', 'CN_Ratio', 'Bulk_Density']
Đang xử lý nhóm: Vegetation_Health_Index <- ['EVI', 'NDVI_Season_Min', 'NDVI_Season_CV', 'Combined_NDVI_Seaso_LAI_NDVI_Seaso_FPAR', 'Combined_NDVI_Season_Range_Std']
Đang xử lý nhóm: Wind_Index <- ['Wind_Mean', 'Wind_Max']
Đang xử lý nhóm: Soil_Environment_Index <- ['Soil_Moisture_mm', 'LST_C']

--- Đang tính toán VIF sau khi xử lý ---
                                              Feature        VIF
0                                  Avg_Salinity_Index  13.872211
9                        Combined_Rain_Temp__Rainfall   9.813699
5                           Combined_Temp_Min_Max_Avg   8.281235
1                                                  pH   6.454444
7                        Combined_sm_rootzone_surface   6.234316
6       Combined_Humidity_Avg_MaxRelative_MinRelative   5.83